# Supervised Fine-Tuning (SFT) with Serverless Customization on SageMaker AI

## Lab 1 — Contract Review That Cites Its Evidence: Prepare the Data

This is the first of four interconnected labs:

| Lab | Notebook | What you'll do |
|-----|----------|----------------|
| **Lab 1** | 1-prepare-data.ipynb ← *you are here* | Download ContractNLI, build training records, and register datasets |
| **Lab 2** | 2-fine-tune-llm.ipynb | Submit a serverless LoRA fine-tuning job and register the result |
| **Lab 3** | 3-evaluation.ipynb | Score the base, frontier, and fine-tuned model on held-out contracts |
| **Lab 4** | 4-deployment.ipynb | Deploy the merged model to a real-time SageMaker endpoint |

### What you'll do in this lab

1. **Download the dataset** — [ContractNLI](https://stanfordnlp.github.io/contract-nli/) (CC-BY-4.0), 607 real NDAs annotated against a fixed legal checklist
2. **Build training records** — one prompt/completion pair per contract, with the prompt identical to what the model sees at inference time
3. **Upload** the three splits (train / val / test) to Amazon S3
4. **Register** them as versioned DataSet assets in the SageMaker AI Registry

---


### The Task

[ContractNLI](https://stanfordnlp.github.io/contract-nli/) is a **document-level Natural Language Inference (NLI)** task for automated contract review. Given an NDA and a fixed set of 17 hypotheses, the model must:

1. **Classify** each hypothesis as Entailment, Contradiction, or NotMentioned relative to the contract
2. **Identify evidence** — the specific numbered spans (clauses) that justify each Entailment or Contradiction decision

<img src="images/hypothesis_example.png" width="720">

Each hypothesis in the checklist is a statement about what a contract *should* say. For each one, the model reads the full contract and decides:

- **Entailment** — the contract explicitly states or implies the hypothesis is true. A clause directly supports it, and the model must cite that clause number.
- **Contradiction** — the contract says something that *conflicts* with the hypothesis. This is subtler than just "false" — it means the contract actively says the opposite. A broad prohibition with no exception can contradict a hypothesis even if the hypothesis topic isn't mentioned by name.
- **NotMentioned** — the contract simply doesn't address the hypothesis at all. No clause supports it, no clause conflicts with it.

The distinction between Contradiction and NotMentioned is what makes this hard. When a contract is silent on something it's NotMentioned, but when it contains a broad prohibition that leaves no room for the hypothesis to be true, that's a Contradiction. The model has to read carefully to tell the difference.

In the dataset, each contract/hypothesis pair is annotated with:

```json
{
  "choice": "Entailment | Contradiction | NotMentioned",
  "spans": [3, 4]  // clause indices that justify the decision; empty for NotMentioned
}
```

A model does this for all 17 hypotheses at once, in a single reply. That reply is what the next section shows.

### The Expected Output

The model responds with strict JSON — one entry per hypothesis, each with a label and evidence spans. Each key such as nda-11 identifies one of the 17 checklist hypotheses, not the contract — the same 17 keys appear in the output for every contract, since every contract is checked against the same checklist. The keys run from 1 to 20 rather than 1 to 17: three numbers (6, 9, and 14) don't exist in ContractNLI's original numbering, so 17 hypotheses land on IDs that reach 20.

```json
{
  "nda-11": {"label": "NotMentioned",  "evidence": []},
  "nda-16": {"label": "NotMentioned",  "evidence": []},
  "nda-15": {"label": "NotMentioned",  "evidence": []},
  "nda-10": {"label": "NotMentioned",  "evidence": []},
  "nda-2":  {"label": "Contradiction", "evidence": [3, 4]},
  "nda-1":  {"label": "NotMentioned",  "evidence": []},
  "nda-19": {"label": "NotMentioned",  "evidence": []},
  "nda-12": {"label": "NotMentioned",  "evidence": []},
  "nda-20": {"label": "NotMentioned",  "evidence": []},
  "nda-3":  {"label": "NotMentioned",  "evidence": []},
  "nda-18": {"label": "NotMentioned",  "evidence": []},
  "nda-7":  {"label": "Contradiction", "evidence": [3]},
  "nda-17": {"label": "NotMentioned",  "evidence": []},
  "nda-8":  {"label": "NotMentioned",  "evidence": []},
  "nda-13": {"label": "NotMentioned",  "evidence": []},
  "nda-5":  {"label": "Contradiction", "evidence": [3]},
  "nda-4":  {"label": "Entailment",    "evidence": [4]}
}
```

Producing this JSON for a held-out contract, reliably, is the whole goal of fine-tuning. The rest of this lab builds the training data that teaches a model to do it.

### The Data

The training data for this task is [ContractNLI](https://stanfordnlp.github.io/contract-nli/) (Koreeda & Manning, *Findings of EMNLP 2021*) — 607 real NDAs sourced from SEC filings and the public web, each annotated against the same 17 hypotheses shown above. Released under **CC-BY-4.0**.

| Split | Contracts | Used for |
|-------|-----------|---------|
| Train | 423 | Fine-tuning |
| Dev | 61 | Validation during training |
| Test | 123 | Held-out evaluation in Lab 3 |

Splits are document-level — no contract appears in more than one split. The rest of this notebook turns these 607 contracts into prompt/completion records shaped exactly like the JSON above, then uploads and registers them so Lab 2 can train on them.

---

### Install requirements

In [ ]:
%pip install -r requirements.txt

  Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached numpy-2.2.6-cp312-cp312-macosx_14_0_arm64.whl.metadata (62 kB)
Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl (10.7 MB)
Using cached numpy-2.2.6-cp312-cp312-macosx_14_0_arm64.whl (5.1 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.2
    Uninstalling numpy-2.5.2:
      Successfully uninstalled numpy-2.5.2
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.5
    Uninstalling pandas-3.0.5:
      Successfully uninstalled pandas-3.0.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
streamlit 1.37.1 requires cachetools<6,>=4.0, but yo

### Set up the SageMaker session

We start by creating a **SageMaker Session** — a lightweight helper that manages the connection to your AWS account. It:
- Resolves the **default S3 bucket** for staging datasets and model artifacts
- Reads your **IAM execution role** — the identity that grants SageMaker permission to read S3, write metrics, and launch training jobs on your behalf

> **Tip:** If you're running inside SageMaker Studio, get_execution_role() automatically retrieves the Studio execution role. Outside Studio, you can create a role named sagemaker_execution_role in IAM with the AmazonSageMakerFullAccess managed policy attached.


In [ ]:
import os
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"  # pin ambient region: sagemaker.ai_registry.AIRHub
                                                  # hashes boto3's *default* session region into its
                                                  # private hub name, ignoring any sagemaker_session
                                                  # passed explicitly, so this must be set before import
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role
import numpy
print(numpy.__file__)

boto_sess = boto3.Session()
sess = Session(boto_session=boto_sess)
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(boto_session=boto_sess, default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/clockhart/Library/Application Support/sagemaker/config.yaml


[08/24/26 16:34:03] INFO     Loading cached SSO token for nvsec-nv-aws-mpa                            ]8;id=529664;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=754611;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py#377\377]8;;\

                    INFO     SSO Token refresh succeeded                                              ]8;id=813277;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=786740;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py#347\347]8;;\

sagemaker role arn: arn:aws:iam::492681118881:role/service-role/AmazonSageMaker-ExecutionRole-20201215T102238
sagemaker bucket: clockhart-bucket
sagemaker session region: us-east-1


### contractnli.py

Every notebook starts with import contractnli as C, a helper module for downloading data, reading it, and rendering the prompt. These are the helper functions:

| Call | Returns | Used for |
|---|---|---|
| C.ensure_dataset("./data") | the unpacked path | downloads the ContractNLI archive once |
| C.load(split) | documents, checklist | reads train, dev or test |
| C.doc_spans(doc) | numbered clause list | one contract as numbered clauses |
| C.gold_for(doc) | expert answer dict | the expert answer for one contract |
| C.build_prompt(doc, labels) | one string | the whole request: instruction, contract, checklist |

One constant matters too: C.INSTRUCTION, the template build_prompt fills in — you'll print it below.

### Step 1 – Download the dataset

The checklist is identical across all three splits, so only the train copy is kept as labels below, and that one dict renders the checklist into every prompt.

In [4]:
import contractnli as C

C.ensure_dataset("./data")

train_docs, labels = C.load("train")
dev_docs, _ = C.load("dev")
test_docs, _ = C.load("test")

print(f"train {len(train_docs)} contracts | dev {len(dev_docs)} | test {len(test_docs)}")
print(f"checklist items: {len(labels)}")

train 423 contracts | dev 61 | test 123
checklist items: 17


#### What one document actually looks like

Before building anything, it is worth seeing the raw shape you are working from. The
dataset gives you documents; the three helpers below are how you get from a document to
the pieces the prompt needs.


In [ ]:
doc_example = train_docs[0]

print("One ContractNLI document is a plain dict. Its fields:\n")
for key, value in doc_example.items():
    size = f"{len(value):,} items" if isinstance(value, list) else f"{len(str(value)):,} chars"
    print(f"  doc[{key!r}]:{' ' * (20 - len(key))}{type(value).__name__:5s} {size}")

print("\nOnly three of those matter here, and `contractnli.py` has a helper for each.\n")

# 1. The contract, cut into the clauses the model will cite by number.
print("1. doc['spans'] holds (start, end) offsets into doc['text'], the dataset's own")
print("   clause split. C.doc_spans(doc) slices them out and numbers them:\n")
for number, text in C.doc_spans(doc_example)[:3]:
    print(f"     [{number}] {text[:62]}")

# 2. The expert answer. `choice` is the verdict, `spans` the clauses that justify it.
print("\n2. doc['annotation_sets'] holds the expert labels. C.gold_for(doc) unwraps it")
print("   to one entry per checklist item:\n")
gold_example = C.gold_for(doc_example)
for key in list(labels)[:2]:
    print(f"     {key}: {gold_example[key]}")

# 3. The checklist is the same for every contract. In this format it is rendered into
#    every prompt, which is why it comes from one dict rather than per-record text.
print("\n3. `labels`, the second value C.load() returned, is the checklist itself:\n")
first_key = list(labels)[0]
print(f"     labels[{first_key!r}]:")
for field, text in labels[first_key].items():
    print(f"       {field}: {text[:66]}")

One ContractNLI document is a plain dict. Its fields:

  doc['id']:                  int   2 chars
  doc['file_name']:           str   56 chars
  doc['text']:                str   8,585 chars
  doc['spans']:               list  65 items
  doc['annotation_sets']:     list  1 items
  doc['document_type']:       str   10 chars
  doc['url']:                 str   73 chars

Only three of those matter here, and `contractnli.py` has a helper for each.

1. doc['spans'] holds (start, end) offsets into doc['text'] — the dataset's own
   clause split. C.doc_spans(doc) slices them out and numbers them:

     [0] NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT
     [1] This NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT (“Agreement”
     [2] (i) the Office of the United Nations High Commissioner for Ref

2. doc['annotation_sets'] holds the expert labels. C.gold_for(doc) unwraps it
   to one entry per checklist item:

     nda-11: {'choice': 'NotMentioned', 'spans': []}
     nda-16: {'choice': 'Entailmen

### The checklist the model has to answer

The 17 hypotheses, each with the short description that names it. build_prompt() renders these into the checklist block of the prompt, so every contract is judged against this same list.

The sort key below is for readability only: the dataset's numbering starts at nda-11, and plain sorting would put nda-10 before nda-2. Numbers 6, 9 and 14 are unused, which is why 17 items reach nda-20.

In [6]:
for k, v in sorted(labels.items(), key=lambda kv: int(kv[0].split("-")[1])):
    print(f"{k:7s} [{v['short_description']}]")
    print(f"        {v['hypothesis']}")

nda-1   [Explicit identification]
        All Confidential Information shall be expressly identified by the Disclosing Party.
nda-2   [None-inclusion of non-technical information]
        Confidential Information shall only include technical information.
nda-3   [Inclusion of verbally conveyed information]
        Confidential Information may include verbally conveyed information.
nda-4   [Limited use]
        Receiving Party shall not use any Confidential Information for any purpose other than the purposes stated in Agreement.
nda-5   [Sharing with employees]
        Receiving Party may share some Confidential Information with some of Receiving Party's employees.
nda-7   [Sharing with third-parties]
        Receiving Party may share some Confidential Information with some third-parties (including consultants, agents and professional advisors).
nda-8   [Notice on compelled disclosure]
        Receiving Party shall notify Disclosing Party in case Receiving Party is required by law, regu

### Look at one real contract

Below is one of the shortest NDAs in the test set, split into numbered spans, followed by the gold answer. Read span [3], then look at nda-5 (sharing with employees) and nda-7 (sharing with third parties).

In [7]:
doc = sorted(test_docs, key=lambda d: len(d["text"]))[1]
spans = C.doc_spans(doc)

print(f"{doc['file_name']}  —  {len(doc['text'].split())} words, {len(spans)} spans\n")
for i, t in spans:
    print(f"[{i}] {t[:160]}")

1023734_0000912057-96-023266_document_16.txt  —  232 words, 13 spans

[0] NAVIDEC, INCORPORATED
[1] TRADE SECRET/NON-DISCLOSURE AGREEMENT
[2] In consideration of the mutual promises made herein, as well as the agreement between Navidec, Incorporated and _______________ , the parties hereby agree as fo
[3] ____________________ , agrees that, in consideration for being shown or told about certain trade secrets or property belonging to Navidec, Incorporated, _______
[4] Further, ___________________ , agrees not to use, either directly or indirectly any of the material, ideas, objects or portions thereof of said trade secret or 
[5] Any dispute that arises hereunder shall be resolved by arbitration pursuant to the rules of the American Arbitration Association or the rules of the State of Co
[6] In the event that any litigation or arbitration is commenced to enforce any of the provisions of this agreement, the prevailing party of said litigation shall b
[7] This agreement shall be governed 

The gold answer for the same contract, one line per checklist item. `C.gold_for()`
returns the `annotations` of the document's single annotation set: per item, a `choice`
from the three labels and `spans`, the span numbers the annotator pointed to, in the
same numbering as the `[i]` markers above. `spans` is non-empty for exactly the
`Entailment` and `Contradiction` entries, in all 10,319 judgements across the three
splits, so the citation rule the prompt states is one the data already obeys.

`gold_json()` further down turns this into the completion, renaming `choice` to
`label` and `spans` to `evidence`. It emits the items in the checklist's order rather
than the numeric order sorted here, which is why the JSON in the introduction starts at
`nda-11`.

In [8]:
gold = C.gold_for(doc)
print("gold answer:\n")
for k in sorted(gold, key=lambda x: int(x.split("-")[1])):
    v = gold[k]
    ev = f"  evidence={v['spans']}" if v["spans"] else ""
    print(f"{k:7s} {v['choice']:14s}{ev}   [{labels[k]['short_description']}]")

gold answer:

nda-1   NotMentioned     [Explicit identification]
nda-2   Contradiction   evidence=[3, 4]   [None-inclusion of non-technical information]
nda-3   NotMentioned     [Inclusion of verbally conveyed information]
nda-4   Entailment      evidence=[4]   [Limited use]
nda-5   Contradiction   evidence=[3]   [Sharing with employees]
nda-7   Contradiction   evidence=[3]   [Sharing with third-parties]
nda-8   NotMentioned     [Notice on compelled disclosure]
nda-10  NotMentioned     [Confidentiality of Agreement]
nda-11  NotMentioned     [No reverse engineering]
nda-12  NotMentioned     [Permissible development of similar information]
nda-13  NotMentioned     [Permissible acquirement of similar information]
nda-15  NotMentioned     [No licensing]
nda-16  NotMentioned     [Return of confidential information]
nda-17  NotMentioned     [Permissible copy]
nda-18  NotMentioned     [No solicitation]
nda-19  NotMentioned     [Survival of obligations]
nda-20  NotMentioned     [Permissible po

### Check the answer against the clause you just read

This is the contract from the introduction, and span [3] is that blanket
prohibition. Notice it drives three separate `Contradiction` verdicts:

| Item | Subject | Evidence |
|---|---|---|
| `nda-5` | sharing with employees | `[3]` |
| `nda-7` | sharing with third parties | `[3]` |
| `nda-2` | only technical information is confidential | `[3, 4]` |

One clause, three verdicts, which is why the model has to reason over the whole
document per item rather than retrieve one passage per question.

Notice also that 13 of the 17 items are `NotMentioned`. Short NDAs are silent on most of
the checklist, and that skew is why accuracy alone is a weak metric here: over the whole
test split 43% of decisions are `NotMentioned`, so always answering it scores 43% without
reading anything, see notebook 3.


### The prompt

Everything the model will ever see is one string: the template below with three
slots filled in, the number of checklist items, the contract as numbered spans, and the
checklist itself.

**The format: chat completion.** Serverless customization accepts a record as a
`prompt`/`completion` pair, which is the shape this notebook writes. Everything the
model reads goes in `prompt`, and the single thing it must produce goes in `completion`;
there are no roles and no turn boundaries to get right. The order inside the prompt is
instruction, then contract, then checklist, then the required output shape, the
checklist sits *after* the contract so the last thing the model reads before answering
is what it is being asked.

It is defined **once**, in `contractnli.py`, and used by every notebook, data prep
here, the frontier baseline in notebook 3, and the serving checks in 4 and 4a. That is
deliberate: the promise that *the training prompt is byte-identical to the inference
prompt* only holds if there is exactly one copy. A second copy pasted into a notebook is
how train/serve skew gets introduced.

In [ ]:
# The exact template. {n}, {spans} and {checklist} are the only substitutions.
# build_prompt() can append C.NO_THINK, but this lab passes no_think=False
# everywhere: Nemotron ignores the directive, so the dataset carries the fix.
print("=" * 70, "\nINSTRUCTION\n", "=" * 70, sep="")
print(C.INSTRUCTION)

INSTRUCTION
You are a contract review assistant. You review a non-disclosure agreement (NDA) against a fixed checklist of {n} legal hypotheses.

For EACH hypothesis, decide:
- "Entailment": the contract states or implies the hypothesis is true.
- "Contradiction": the contract states something that conflicts with the hypothesis.
- "NotMentioned": the contract does not address it.

Also cite the span numbers that justify the decision (the exact spans a lawyer would point to). Cite spans only for Entailment or Contradiction; use an empty list for NotMentioned. Read exceptions and carve-outs carefully: a clause with an exception may contradict a hypothesis stated absolutely.

CONTRACT (numbered spans):
{spans}

CHECKLIST:
{checklist}

Respond with JSON only, no other text:
{{"nda-1": {{"label": "Entailment|Contradiction|NotMentioned", "evidence": [span numbers]}}, ...}}
Include an entry for every hypothesis key listed above.

/no_think    <- appended by build_prompt(doc, labels); no_think=

To experiment with the wording, set `C.INSTRUCTION` here and re-run the record build
below, every notebook then picks up your version:

```python
C.INSTRUCTION = """...your wording, keeping {n}, {spans} and {checklist}..."""
```

#### The empty reasoning block

Nemotron 3 Nano is a reasoning model. Its chat template opens `<think>` at the start of
every assistant turn and nothing closes it, so on a 17-item checklist it fills the whole
generation budget reasoning and never reaches the JSON. Every metric then reads 0.

Qwen3 has the same tendency but obeys `/no_think` in the prompt. Nemotron ignores it,
whether it arrives as a system field, in the user turn, or both, so the switch has to go
somewhere the model reads as its own output. Every training completion below therefore
begins with `<think>\n</think>\n`: the model learns to close the block the template
opened, then answer. At inference it emits `</think>` and goes straight into the JSON.

`/no_think` is not used anywhere in this lab, so both builders pass `no_think=False` and
train, validation and test prompts are the same string. See
[`nemotron_support.py`](nemotron_support.py) for the measured numbers and for the
one-line template edit that reaches the same result without touching the dataset.

### Step 2 – Build the training records

One training record = one contract, with all 17 verdicts in the completion. So
423 records, but each carries 17 supervised decisions plus the evidence spans,
which is roughly 7,200 labelled judgements.

**The format: `prompt`/`completion`.** Two string fields per line, no roles:

```json
{"prompt": "You are a contract review assistant... CONTRACT (numbered spans):\n[0] ... CHECKLIST: ...",
 "completion": "<think>\n</think>\n{\"nda-11\": {\"label\": \"NotMentioned\", ...}}"}
```

The recipes accept this alongside the role-tagged `messages` shape, and for a
single-turn task it is the simpler of the two: there is exactly one boundary in the
record, and it is the one the trainer needs. The completion is the only place this
dataset differs from a non-reasoning model's: the empty block, then the answer.

#### Why the test split looks different

Train and validation use `prompt`/`completion`. The **test** split keeps
`query`/`response`, because notebook 3 scores it with `CustomScorerEvaluator`, which
pins the evaluation task to `gen_qa`, a format whose fields are exactly `query`,
`response` and an optional `system`.

The reference answer stays plain JSON: the reasoning block belongs to the model's turn,
not to the gold answer.

```json
{"query":    "You are a contract review assistant... CONTRACT ... CHECKLIST: ...",
 "response": "{\"nda-11\": ...}"}
```

The optional `system` field is unused. An earlier version of this dataset set it to
`/no_think`; removing it scored marginally higher.

### Build the training records

One training record = one contract, with all 17 verdicts in the completion. So
423 records, but each carries 17 supervised decisions plus the evidence spans,
which is roughly 7,200 labelled judgements.

**The format: `prompt`/`completion`.** Two string fields per line, no roles:

```json
{"prompt": "You are a contract review assistant... CONTRACT (numbered spans):\n[0] ... CHECKLIST: ...",
 "completion": "<think>\n</think>\n{\"nda-11\": {\"label\": \"NotMentioned\", ...}}"}
```

The recipes accept this alongside the role-tagged `messages` shape, and for a
single-turn task it is the simpler of the two: there is exactly one boundary in the
record, and it is the one the trainer needs. The completion is the only place this
dataset differs from a non-reasoning model's: the empty block, then the answer.

#### Why the test split looks different

Train and validation use `prompt`/`completion`. The **test** split keeps
`query`/`response`, because notebook 3 scores it with `CustomScorerEvaluator`, which
pins the evaluation task to `gen_qa`, a format whose fields are exactly `query`,
`response` and an optional `system`.

The reference answer stays plain JSON: the reasoning block belongs to the model's turn,
not to the gold answer.

```json
{"query":    "You are a contract review assistant... CONTRACT ... CHECKLIST: ...",
 "response": "{\"nda-11\": ...}"}
```

The optional `system` field is unused. An earlier version of this dataset set it to
`/no_think`; removing it scored marginally higher.

In [ ]:
import json

import nemotron_support as N

label_keys = list(labels.keys())          # the checklist's own order


def gold_json(doc):
    """The expert answer for one contract, as the exact JSON the model must emit.
    Only the field names change: `choice` becomes `label`, `spans` becomes `evidence`.
    """
    g = C.gold_for(doc)
    return json.dumps({k: {"label": g[k]["choice"], "evidence": list(g[k]["spans"])}
                       for k in label_keys if k in g})


def make_records(docs):
    """Training records: one prompt/completion pair per contract. `training_record`
    prefixes the completion with the empty reasoning block."""
    return [N.training_record(C.build_prompt(d, labels, no_think=False), gold_json(d))
            for d in docs]


def make_test_records(docs):
    """Evaluation records. The managed scorer reads `query`/`response` (genqa) and
    nothing else, so the test split keeps that shape, see the note below. The
    reference answer is plain JSON, with no block."""
    return [N.eval_record(C.build_prompt(d, labels, no_think=False), gold_json(d))
            for d in docs]


records = {"train": make_records(train_docs),
           "val": make_records(dev_docs),
           "test": make_test_records(test_docs)}

for name, rows in records.items():
    field = "query" if name == "test" else "prompt"
    avg = sum(len(r[field]) for r in rows) // len(rows)   # chars, not tokens
    print(f"{name:5s}: {len(rows):4d} records, avg prompt {avg:6d} chars "
          f"(~{avg // 4} tokens)")


the dataset stores:       {"choice": "Entailment", "spans": [39, 40]}
the prompt asks for:      {"label": "Entailment", "evidence": [39, 40]}
                          ^ choice -> label, spans -> evidence, for all 17 items

C.build_prompt(doc, labels) 12,121 chars   the whole request
  of which the contract       8,887 chars   the only part that varies
  fixed instruction etc.      3,234 chars   identical every time
gold_json(doc)                  918 chars   what the model must produce

record keys: ['prompt', 'completion']

One function, two uses: the same build_prompt() output is the `prompt` of a
training record here and the request sent at inference time in notebooks 3 and 4.
That is why they cannot drift apart.


One complete training record, the prompt the model reads and the JSON it must
produce. The cell picks the train record with the **shortest** prompt: 4,783 characters,
of which the contract is only 1,560 over 18 spans, against a median prompt of ~13,554,
so one whole contract fits on screen. It is the smallest NDA in the split, not a typical
one. The completion is stored as a single 912-character line, the 17-character reasoning
block plus 895 of JSON. `indent=1` reformats it for reading here and `[:800]` trims the
display, not the record: all 17 verdicts are in the target.

In [ ]:
sample = min(records["train"], key=lambda r: len(r["prompt"]))

print("=" * 70, "\nPROMPT\n", "=" * 70, sep="")
print(sample["prompt"])
print("\n" + "=" * 70, "\nCOMPLETION\n", "=" * 70, sep="")
print(sample["completion"][:len(N.EMPTY_REASONING)], end="")
answer = sample["completion"][len(N.EMPTY_REASONING):]
print(json.dumps(json.loads(answer), indent=1)[:800], "...")

#### Write to disk and upload to Amazon S3

`shutil.rmtree` removes `./sft_data` first, so a re-run cannot leave a stale file
behind. Each split is then written as `./sft_data/<split>/dataset.jsonl`, one JSON
object per line, the shape `DataSet.create` validates before registering it below. The
directory names are the keys of `records`, so the `dev` documents are written as `val`,
and that is the name notebook 2 fetches the validation set by. The train file comes to
6.8 MB over its 423 lines, about 16 KB a record.

The key joins `DATASET_PREFIX` (`contractnli-nda-review`, from `config.py`) to
`default_prefix`, which the setup cell read from `sess.default_bucket_prefix`, a
leading key segment when a SageMaker config sets one, and `None` otherwise, which is why
the key is assembled with a conditional. Each file lands at
`s3://<bucket>/[<prefix>/]datasets/contractnli-nda-review/<split>/dataset.jsonl`, and
the three URIs accumulate in `s3_paths`, the dict the next cell registers.

In [ ]:
import pathlib
import shutil

from config import DATASET_PREFIX

local = pathlib.Path("./sft_data")
if local.exists():
    shutil.rmtree(local)

for name, rows in records.items():
    d = local / name
    d.mkdir(parents=True, exist_ok=True)
    with open(d / "dataset.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

input_path = (f"{default_prefix}/datasets/{DATASET_PREFIX}" if default_prefix
              else f"datasets/{DATASET_PREFIX}")

s3_paths = {}
for name in records:
    key = f"{input_path}/{name}/dataset.jsonl"
    s3_client.upload_file(str(local / name / "dataset.jsonl"), bucket_name, key)
    s3_paths[name] = f"s3://{bucket_name}/{key}"
    print(s3_paths[name])

#### Register the datasets

`DataSet.create` writes a registry entry pointing at the S3 object you just uploaded,
and what that buys you is a name. Notebook 2 calls
`DataSet.get(name=f"{DATASET_PREFIX}-train")` and hands the returned object to
`SFTTrainer(training_dataset=...)`, which resolves it to the entry's name and version
rather than an S3 URI, so the job records which dataset it consumed. Both evaluators in
notebook 3 resolve the test entry the same way. `wait=True` blocks until each import
reaches `Available`, and raises if it reaches `ImportFailed` instead.

| dataset | technique | consumed by |
|---|---|---|
| `contractnli-nda-review-train` | `SFT` | notebook 2, `training_dataset=` |
| `contractnli-nda-review-val` | `SFT` | notebook 2, `validation_dataset=` |
| `contractnli-nda-review-test` | none | notebook 3, `dataset=` |

The technique is stored as the search keyword `customization_technique:sft`, a label to
find the dataset by, not a constraint; nothing reads it back at training time, since the
trainer knows its own technique. The enum offers `SFT`, `DPO` and `RLVR` and nothing for
evaluation data, so the test split is registered without one.

`create` also downloads each file and matches it against the formats the registry
knows (the prompt/completion shape for train and val, `genqa` for the test split) so
the wrong shape fails here rather than inside a training job. It reads only the file's first
record.
The match is pass/fail and no format name is stored anywhere in the entry, which is why
the registry console shows an empty **Format** column for every dataset, it is not a
sign that anything is wrong with yours.

> **Note:** `create` overwrites nothing, it reads the current version
> and imports the next major one, so a second run leaves `2.0.0` beside `1.0.0`, and a
> lookup by bare name returns the latest. But the entry records only a bucket and a key,
> with no version id or checksum, and the cell above always writes the same key. Every
> version therefore resolves to the *same* `dataset.jsonl`: the version list is a
> history of registrations, not of your data. Re-upload before you re-register, and
> never expect `1.0.0` to still hold the records it was created with.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique


def register(name, source, technique=None):
    kwargs = dict(name=name, source=source, wait=True, role=role, sagemaker_session=sess)
    if technique is not None:
        kwargs["customization_technique"] = technique
    ds = DataSet.create(**kwargs)
    print(f"created dataset: {name}")
    return ds


training_dataset = register(f"{DATASET_PREFIX}-train", s3_paths["train"], CustomizationTechnique.SFT)
val_dataset = register(f"{DATASET_PREFIX}-val", s3_paths["val"], CustomizationTechnique.SFT)
test_dataset = register(f"{DATASET_PREFIX}-test", s3_paths["test"])

### What you built

`contractnli-nda-review-train` (423 records), `-val` (61) and `-test` (123 held out),
registered and ready. Notebooks 2 and 3 look them up by name, no variable defined in
this notebook has to survive.

Continue to **notebook 2** to run the serverless LoRA fine-tuning job